# Optional: Gemini via API (not completed)

Frontier-LLM comparison through the Gemini API. It needs a key in Colab Secrets as GEMINI_API_KEY. Not completed in this project because of free-tier quota and availability limits; the code is included for anyone with API access.

In [ ]:
# ===== STEP 5: frontier LLM (Google Gemini, free tier) on the same 1,000 sentences =====
# Needs a free key from aistudio.google.com saved in Colab Secrets as GEMINI_API_KEY (Notebook access ON).
# CPU runtime is fine for this step.
!pip -q install -U google-genai

import os, json, time, re
import numpy as np, pandas as pd
from google import genai
from google.genai import types
from google.colab import drive, userdata
from sklearn.metrics import f1_score, accuracy_score, classification_report
drive.mount("/content/drive")

DATA = "/content/drive/MyDrive/sci-classify/data"; RES = "/content/drive/MyDrive/sci-classify/results"
LABELS = ["BACKGROUND", "OBJECTIVE", "METHODS", "RESULTS", "CONCLUSIONS"]
test_llm = pd.read_parquet(f"{DATA}/test_llm_1000.parquet")
train = pd.read_parquet(f"{DATA}/train.parquet")

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))   # the key is read from Secrets, never typed here

# Pick a current "Flash" model available to free keys (names change over time, so we look them up)
names = [m.name.replace("models/", "") for m in client.models.list()]
flash = [n for n in names if "flash" in n and not any(x in n for x in
         ["image", "tts", "audio", "live", "thinking", "exp", "embedding", "lite", "preview", "omni", "latest"])]

def version(n):                       # "gemini-3.7-flash" -> (3, 7); newest first
    m = re.search(r"(\d+)\.(\d+)", n)
    return (int(m.group(1)), int(m.group(2))) if m else (0, 0)

# Pin ONE model for the whole experiment. Busy errors (503) are temporary, so we wait them out;
# only a model that no longer exists (404 / not found) is skipped. Set PINNED to force a specific model.
PINNED = "gemini-3.8-flash"
MODEL = None
for cand in ([PINNED] if PINNED in flash else []) + sorted(flash, key=version, reverse=True):
    try:
        client.models.generate_content(model=cand, contents="Reply with the word OK.")
        MODEL = cand; break
    except Exception as e:
        msg = str(e)
        if "404" in msg or "NOT_FOUND" in msg or "no longer available" in msg:
            print(f"  {cand} not available: {msg[:80]}"); continue
        MODEL = cand                                   # exists but busy: keep it, the retries will wait
        print(f"  {cand} is busy right now ({msg[:40]}...) - keeping it and waiting through retries"); break
print("Stable Flash models found:", flash); print("Using:", MODEL)
assert MODEL, "No Flash model available - paste this output to Claude"

BATCH = 20          # sentences per request: keeps us well inside free-tier request limits
PAUSE = 7           # seconds between requests: stays under free-tier per-minute limits

INSTRUCTIONS = (
    "You classify sentences from abstracts of randomised controlled trials by their rhetorical role.\n"
    "Labels:\n"
    "BACKGROUND: context, prior knowledge or the problem being addressed.\n"
    "OBJECTIVE: the aim, purpose or hypothesis of this study.\n"
    "METHODS: design, participants, interventions, measurements or analysis.\n"
    "RESULTS: findings, numbers and statistical outcomes.\n"
    "CONCLUSIONS: interpretation, implications or recommendations.\n"
    "Return only a JSON list of labels, one per sentence, in the same order.")

def few_shot_block():
    ex = train.groupby("label").sample(n=2, random_state=7)
    lines = [f'[{r.line_number + 1} of {r.total_lines}] {r.text} -> {r.label}' for r in ex.itertuples()]
    return "Examples:\n" + "\n".join(lines) + "\n\n"

def build_prompt(rows, use_position, few_shot):
    items = [f"{i + 1}. " + (f"[{r.line_number + 1} of {r.total_lines}] " if use_position else "") + r.text
             for i, r in enumerate(rows.itertuples())]
    pos_note = "Each sentence starts with its position in its abstract, e.g. [3 of 9].\n" if use_position else ""
    return INSTRUCTIONS + "\n" + pos_note + "\n" + (few_shot_block() if few_shot else "") + \
           "Sentences:\n" + "\n".join(items)

def ask(prompt, n, tries=8):
    for attempt in range(tries):
        try:
            r = client.models.generate_content(
                model=MODEL, contents=prompt,
                config=types.GenerateContentConfig(temperature=0, response_mime_type="application/json"))
            out = json.loads(r.text)
            out = [str(x).strip().upper() for x in out]
            if len(out) == n:
                return out
            print(f"  wrong length ({len(out)} for {n}), retrying")
        except Exception as e:
            wait = min(60 * (attempt + 1), 300)       # 1, 2, 3 ... up to 5 minutes between retries
            print(f"  error: {str(e)[:120]} - waiting {wait}s")
            time.sleep(wait)
    return ["INVALID"] * n

def run(name, use_position, few_shot):
    path = f"{RES}/step5_{name}_{MODEL}.json"          # one file per model, so failed runs never mix in
    done = json.load(open(path)) if os.path.exists(path) else {}       # resumable: skips finished batches
    done = {k: v for k, v in done.items() if v and v[0] != "INVALID"}  # batches that failed (e.g. server overload) are retried
    t0 = time.time()
    for start in range(0, len(test_llm), BATCH):
        key = str(start)
        if key in done:
            continue
        rows = test_llm.iloc[start:start + BATCH]
        answer = ask(build_prompt(rows, use_position, few_shot), len(rows))
        if answer[0] == "INVALID":
            print(f"\n  batch {start} skipped for now (server busy) - it will be retried when you rerun the cell")
            continue
        done[key] = answer
        json.dump(done, open(path, "w"))
        print(f"{name}: {min(start + BATCH, len(test_llm))}/{len(test_llm)}", end="\r")
        time.sleep(PAUSE)
    missing = [k for k in range(0, len(test_llm), BATCH) if str(k) not in done]
    if missing:
        print(f"\n{name}: {len(missing)} batches still missing (server busy). Rerun this cell later to finish them.")
        return None
    preds = [p for k in sorted(done, key=int) for p in done[k]]
    valid = [p if p in LABELS else "INVALID" for p in preds]
    f1 = f1_score(test_llm.label, valid, average="macro", labels=LABELS)
    print(f"\n{name:28s} macro-F1 {f1:.3f}  accuracy {accuracy_score(test_llm.label, valid):.3f}  "
          f"invalid {valid.count('INVALID')}  ({time.time() - t0:.0f}s this session)")
    return valid

results = {}
for name, pos, fs in [("gemini_zero_shot", False, False),
                      ("gemini_zero_shot_position", True, False),
                      ("gemini_few_shot_position", True, True)]:
    r = run(name, pos, fs)
    if r is not None:
        results[name] = r

if not results:
    raise SystemExit("No condition finished yet - rerun this cell later to continue.")
best = max(results, key=lambda k: f1_score(test_llm.label, results[k], average="macro", labels=LABELS))
print(f"\nPer-class F1 for the best Gemini setting ({best}):")
print(classification_report(test_llm.label, results[best], labels=LABELS, digits=3, zero_division=0))
json.dump({"model": MODEL, "preds": results}, open(f"{RES}/step5_gemini_all.json", "w"))
print("Saved to", RES)
